# 05 — Build leakage-safe pre-game features

To predict a game **before it happens**, every feature must use only information available
**before tip-off**. For each team we compute its **season-to-date** form using a window that
ends at the *previous* game:

```
ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING   (partitioned by team, season, ordered by date)
```

That `1 PRECEDING` is the whole game — it guarantees the current game's result never leaks
into its own features. We then attach the home team's prior form and the away team's prior
form to each game, and add **difference features** (home − away) that the linear model loves.

Per team we carry: games played, win %, average margin, the four offensive + four defensive
Four Factors, average net rating, and rest days (days since last game). Early-season games
(fewer than `MIN_GAMES` played) are dropped because their averages are too noisy.

Output: **`gold.game_features`**, one row per (regular-season) game, label `home_win`.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

GAMES    = f"{UC_CATALOG}.{SILVER_SCHEMA}.games"
TEAM_BOX = f"{UC_CATALOG}.{SILVER_SCHEMA}.team_game_box"
FEATURES = f"{UC_CATALOG}.{GOLD_SCHEMA}.game_features"

REG_ONLY  = os.getenv("NBA_REGULAR_SEASON_ONLY", "true").lower() == "true"
MIN_GAMES = int(os.getenv("NBA_MIN_PRIOR_GAMES", "10"))
PLAYOFF_FILTER = "WHERE NOT is_playoff" if REG_ONLY else ""
GAME_FILTER    = "AND NOT g.is_playoff" if REG_ONLY else ""
print(f"Regular season only: {REG_ONLY} | min prior games: {MIN_GAMES}")
print("Feature table:", FEATURES)

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Regular season only: True | min prior games: 10
Feature table: alexander_booth.basketball_reference_waf_gold.game_features


## Build the feature table

`CREATE OR REPLACE TABLE ... AS` keeps it a clean, idempotent full rebuild.

In [2]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {FEATURES}
    CLUSTER BY (season, game_date)
    COMMENT 'Gold - leakage-safe pre-game features (home/away season-to-date form + diffs), label home_win.'
    AS
    WITH tg AS (
        SELECT team, season, game_date, game_sk, won, margin,
               efg, tov_pct, orb_pct, ft_rate, def_efg, def_tov_pct, drb_pct, def_ft_rate, net_rtg
        FROM {TEAM_BOX}
        {PLAYOFF_FILTER}
    ),
    prior AS (
        SELECT team, season, game_sk,
            COUNT(*)                                  OVER w  AS games_played,
            AVG(CASE WHEN won THEN 1 ELSE 0 END)      OVER w  AS win_pct,
            AVG(margin)      OVER w  AS avg_margin,
            AVG(efg)         OVER w  AS efg_for,
            AVG(tov_pct)     OVER w  AS tov_for,
            AVG(orb_pct)     OVER w  AS orb_for,
            AVG(ft_rate)     OVER w  AS ftr_for,
            AVG(def_efg)     OVER w  AS efg_against,
            AVG(def_tov_pct) OVER w  AS tov_against,
            AVG(drb_pct)     OVER w  AS drb_against,
            AVG(def_ft_rate) OVER w  AS ftr_against,
            AVG(net_rtg)     OVER w  AS net_rtg_avg,
            datediff(game_date, lag(game_date) OVER w2) AS rest_days
        FROM tg
        WINDOW
            w  AS (PARTITION BY team, season ORDER BY game_date, game_sk
                   ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING),
            w2 AS (PARTITION BY team, season ORDER BY game_date, game_sk)
    )
    SELECT
        g.game_sk, g.season, g.game_date, CAST(g.home_win AS INT) AS home_win,
        h.games_played AS home_gp, a.games_played AS away_gp,
        COALESCE(h.rest_days, 3) AS home_rest, COALESCE(a.rest_days, 3) AS away_rest,
        -- difference features (home minus away); these are what we feed the model
        (h.win_pct      - a.win_pct)      AS win_pct_diff,
        (h.avg_margin   - a.avg_margin)   AS margin_diff,
        (h.net_rtg_avg  - a.net_rtg_avg)  AS net_rtg_diff,
        (h.efg_for      - a.efg_for)      AS efg_for_diff,
        (h.tov_for      - a.tov_for)      AS tov_for_diff,
        (h.orb_for      - a.orb_for)      AS orb_for_diff,
        (h.ftr_for      - a.ftr_for)      AS ftr_for_diff,
        (h.efg_against  - a.efg_against)  AS efg_against_diff,
        (h.tov_against  - a.tov_against)  AS tov_against_diff,
        (h.drb_against  - a.drb_against)  AS drb_against_diff,
        (h.ftr_against  - a.ftr_against)  AS ftr_against_diff,
        (COALESCE(h.rest_days,3) - COALESCE(a.rest_days,3)) AS rest_diff,
        -- keep the raw home/away form too, for inspection / dashboards
        h.win_pct AS home_win_pct, a.win_pct AS away_win_pct,
        h.net_rtg_avg AS home_net_rtg, a.net_rtg_avg AS away_net_rtg
    FROM {GAMES} g
    JOIN prior h ON h.game_sk = g.game_sk AND h.team = g.home_team
    JOIN prior a ON a.game_sk = g.game_sk AND a.team = g.away_team
    WHERE h.games_played >= {MIN_GAMES} AND a.games_played >= {MIN_GAMES}
      {GAME_FILTER}
""")

try:
    spark.sql(f"ALTER TABLE {FEATURES} ALTER COLUMN game_sk SET NOT NULL")
    spark.sql(f"ALTER TABLE {FEATURES} DROP CONSTRAINT IF EXISTS game_features_pk")
    spark.sql(f"ALTER TABLE {FEATURES} ADD CONSTRAINT game_features_pk PRIMARY KEY (game_sk) RELY")
except Exception as e:
    print("  PK:", str(e)[:80])

print("game_features rows:", spark.table(FEATURES).count())

game_features rows: 3222


## Verify — row counts, label balance, and a leakage guard

In [3]:
spark.sql(f"""
    SELECT season, COUNT(*) AS games,
           ROUND(AVG(home_win), 3) AS home_win_rate,
           ROUND(AVG(margin_diff), 2) AS avg_margin_diff,
           MIN(game_date) AS first_modeled_game
    FROM {FEATURES} GROUP BY season ORDER BY season
""").show()

# Leakage guard: every modeled game must have >= MIN_GAMES of prior history for both teams.
bad = spark.sql(f"SELECT COUNT(*) FROM {FEATURES} WHERE home_gp < {MIN_GAMES} OR away_gp < {MIN_GAMES}").collect()[0][0]
print(f"Rows violating the min-prior-games rule (should be 0): {bad}")

spark.sql(f"""
    SELECT game_date, ROUND(win_pct_diff,3) AS win_pct_diff, ROUND(margin_diff,2) AS margin_diff,
           ROUND(net_rtg_diff,2) AS net_rtg_diff, home_win
    FROM {FEATURES} ORDER BY game_date LIMIT 5
""").show()

+------+-----+-------------+---------------+------------------+
|season|games|home_win_rate|avg_margin_diff|first_modeled_game|
+------+-----+-------------+---------------+------------------+
|  2023| 1073|        0.576|          -0.12|        2022-11-07|
|  2024| 1074|        0.537|          -0.26|        2023-11-14|
|  2025| 1075|        0.542|            0.0|        2024-11-10|
+------+-----+-------------+---------------+------------------+



Rows violating the min-prior-games rule (should be 0): 0


+----------+------------+-----------+------------+--------+
| game_date|win_pct_diff|margin_diff|net_rtg_diff|home_win|
+----------+------------+-----------+------------+--------+
|2022-11-07|         0.1|        4.7|        4.43|       0|
|2022-11-07|        -0.1|        0.9|        1.03|       0|
|2022-11-07|      -0.145|       -6.3|       -5.87|       1|
|2022-11-09|       -0.05|      -3.18|       -2.89|       0|
|2022-11-09|       0.427|      13.13|       13.12|       1|
+----------+------------+-----------+------------+--------+



Features are ready and leakage-free. **Next:** `06_train_with_mlflow` trains three models
and logs them to MLflow.